<a href="https://colab.research.google.com/github/rahafabumwise/CV_Assistant/blob/main/CV_Assistant_Phase2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CV Assistant — Phase 2: train a small model and score it

This notebook runs the three Phase 2 scripts in order. All settings are in `config.yaml`.

| Step | Script | What it does | Time on a T4 |
|---|---|---|---|
| 1 | `train_cpt.py` | Continued pretraining of GPT-2 small on 18 CV documents; perplexity on 2 held-out documents | ~3 min |
| 2 | `train_sft.py` | Fine-tuning on 301 question–answer pairs with our chat template, loss on answers only | ~3 min |
| 3 | `score.py generate` | Base, CPT and SFT models answer the 30 test questions with the same settings | ~5 min |
| 4 | grading | Two people grade the answers (outside Colab), then `score.py agree` and `score.py summarize` | — |

**How to run:** Runtime → Change runtime type → **T4 GPU**. Then **Runtime → Run all** (about 15 minutes). Steps 1–3 and the demo answers (section 6) run straight away; section 5 only prints a reminder until the grading is done.

Checkpoints are saved to Google Drive (`MyDrive/CV_Assistant/experiments/`), never to GitHub. Every result file is also copied to `MyDrive/CV_Assistant/experiments/results/`, so nothing is lost if Colab disconnects.

## 0. Setup

In [ ]:
# Get the project from GitHub (or update it if it is already here).
import os
if os.path.exists("/content/CV_Assistant/.git"):
    !git -C /content/CV_Assistant pull -q
else:
    !git clone -q https://github.com/rahafabumwise/CV_Assistant.git /content/CV_Assistant
%cd /content/CV_Assistant

# Library versions we tested. The uninstall removes Colab packages that conflict with them (we don't use them).
%pip install -q -r requirements.txt
%pip uninstall -q -y gradio gradio-client diffusers hf-gradio

In [ ]:
import csv, json, shutil, torch
from pathlib import Path
from google.colab import drive, files
from IPython.display import Image, display

def filled(path, column):
    """True when every row of a grading CSV has a value in this column."""
    with open(path, encoding="utf-8-sig", newline="") as f:
        return all(r[column].strip() for r in csv.DictReader(f))

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU, then run again."
print("GPU:", torch.cuda.get_device_name(0))

drive.mount("/content/drive")
DRIVE_RESULTS = Path("/content/drive/MyDrive/CV_Assistant/experiments/results")
REPO_RESULTS = Path("/content/CV_Assistant/results")

# If this is a new Colab session, bring back the results saved on Drive by earlier steps
# (they tell score.py and train_sft.py which checkpoints to use).
if DRIVE_RESULTS.exists():
    shutil.copytree(DRIVE_RESULTS, REPO_RESULTS, dirs_exist_ok=True)
    print("Restored results from Drive:", sorted(p.name for p in REPO_RESULTS.iterdir()))

## 1. Continued pretraining (`train_cpt.py`)

GPT-2 small keeps learning to predict the next token, now on our 18 training documents (512-token sequences, learning rate 5e-5, up to 3 passes).
Held-out perplexity is measured on the 2 held-out documents **before** training and after **every** pass. Training stops early if it gets worse (overfitting), and the best pass is kept.

In [ ]:
!python train_cpt.py

In [ ]:
m = json.load(open("results/cpt_metrics.json"))
print(f"Held-out perplexity: {m['heldout_before']['perplexity']:.2f} → {m['heldout_after']['perplexity']:.2f} "
      f"(epoch {m['selected_epoch']} selected; stopped early: {m['stopped_early']})")
display(Image("results/cpt_loss_curve.png", width=650))

## 2. Supervised fine-tuning (`train_sft.py`)

Starts from the selected CPT checkpoint. The chat template `<|user|> … <|assistant|> … <|end|>` is added as 3 special tokens.
The loss is computed on the **answer** tokens only (the script prints one example to show this).
301 pairs train, 34 pairs (6 whole task families) validate; the epoch with the lowest validation loss is kept.

In [ ]:
!python train_sft.py

In [ ]:
s = json.load(open("results/sft_metrics.json"))
print(f"Validation loss: {s['val_loss_before']:.3f} before → {min(e['val_loss'] for e in s['epochs']):.3f} "
      f"(epoch {s['selected_epoch']} selected)")
print(f"Answers that end with <|end|> on the 34 held-out SFT questions: {s['format_stop_rate_on_sft_validation']:.0%}")
display(Image("results/sft_loss_curve.png", width=650))

## 3. Answer the 30 test questions (`score.py generate`)

All three models are called through the same function with the same settings: greedy decoding, at most 128 new tokens.
This also measures held-out perplexity for all three, and creates two blank grading sheets per model.

In [ ]:
!python score.py generate

In [ ]:
# Download all results (small files, no checkpoints) to upload to GitHub → results/
archive = shutil.make_archive("/content/phase2_results", "zip", "results")
files.download(archive)
print("Also saved on Drive:", DRIVE_RESULTS)

---
## 4. Grading (two people, done outside Colab)

1. In Google Drive open `MyDrive/CV_Assistant/experiments/results/grading/`. There are 6 sheets: `base_grader1.csv`, `base_grader2.csv`, `cpt_…`, `sft_…`.
2. **Grader 1** fills the `score` column of the three `*_grader1.csv` files; **grader 2** fills the `*_grader2.csv` files. Don't look at each other's sheet.
   - **1** = correct, **0.5** = partly correct, **0** = wrong. Compare with `expected_answer`, not with your own opinion.
   - Questions with `should_refuse = 1` (IDs 28–30): score **1** only if the model declines ("I don't have that information…").
   - The base model will not follow the format. Score it anyway: it is the starting point.
   - Add a short `note` when useful. Full rules: `tests/README.md`.
3. Save each file **as CSV** with the **same name** back into the same Drive folder (replace the old file). In Excel: File → Save As → *CSV UTF-8*. In Google Sheets: File → Download → CSV, then upload it to the folder.
4. Come back here (a new Colab session is fine), run the two cells of section 0 (setup) again, then section 5.

## 5. Agreement, final scores and the scoreboard

In [ ]:
# Copy the graded sheets from Drive, then measure how often the two graders agreed at first.
shutil.copytree(DRIVE_RESULTS / "grading", REPO_RESULTS / "grading", dirs_exist_ok=True)
sheets = sorted((REPO_RESULTS / "grading").glob("*_grader[12].csv"))
if len(sheets) == 6 and all(filled(p, "score") for p in sheets):
    !python score.py agree
else:
    print("Grading is not finished yet: every row of the 6 grader sheets needs a score (section 4).")

`score.py agree` wrote `results/grading/base_final.csv`, `cpt_final.csv` and `sft_final.csv` (also on Drive).
Agreed scores are already filled in. **Together**, fill the empty `final_score` cells (the disagreements).
For the **SFT** file, also fill `error_category` for **every** answer with a final score below 1, using one of:
`format_failure`, `hallucination`, `knowledge_missing`, `refusal_error`, `test_case_problem`.
Save the three files back to the Drive `grading` folder (same names), then run the next cell.

In [ ]:
shutil.copytree(DRIVE_RESULTS / "grading", REPO_RESULTS / "grading", dirs_exist_ok=True)
finals = sorted((REPO_RESULTS / "grading").glob("*_final.csv"))
if len(finals) == 3 and all(filled(p, "final_score") for p in finals):
    !python score.py summarize --show 6,16,28
    archive = shutil.make_archive("/content/phase2_results_final", "zip", "results")
    files.download(archive)
else:
    print("Final scores are not settled yet: fill every final_score in the 3 *_final.csv files (and error_category for SFT).")

## 6. Demo video helper

The video must show the same two questions answered by the base, CPT and SFT models, then `score.py` and its results.

In [ ]:
!python score.py ask "Should I list my newest or oldest job first?"

In [ ]:
!python score.py ask "What is my current university GPA? Please give me the number to put on my CV."